# Tiny Aya → Wazuh alert analyst (AlignTune LoRA)
Runtime: GPU. v1 ran on a T4; the v2 runs use **A100 + High-RAM** (not a TPU: bitsandbytes and the TRL path are CUDA-only). Before running: accept the licence on https://huggingface.co/CohereLabs/tiny-aya-global and have a Hugging Face **write** token ready.

Set `RUN` in the next cell: `"a"` (15 % injected) or `"b"` (30 % injected). Everything else follows from it. Disconnect the runtime as soon as the export cell finishes.

In [ ]:
RUN = "d"   # "a" (15 % injected), "b" (30 % injected), "c" (a, findings before severity) or "d" (b, plus brute-force-success and rootcheck fixes)
HF_USER = "Yusheng88"
GITHUB_REPO = "https://github.com/Patupi1/wazuh-alert-analyst"
DATASET = f"{HF_USER}/wazuh-alert-analyst-v2"
SUBSET = f"{RUN}_train_messages"
ADAPTER_REPO = f"{HF_USER}/tiny-aya-wazuh-analyst-lora-v2{RUN}"
GGUF_REPO = f"{HF_USER}/tiny-aya-wazuh-analyst-GGUF-v2{RUN}"
# v1 values: DATASET = f"{HF_USER}/wazuh-alert-analyst", SUBSET = "train_messages",
#            ADAPTER_REPO = f"{HF_USER}/tiny-aya-wazuh-analyst-lora", GGUF_REPO = f"{HF_USER}/tiny-aya-wazuh-analyst-GGUF"
# Checkpoints go to Google Drive so they survive a disconnect; the export works on local disk.
from google.colab import drive; drive.mount('/content/drive')
OUT = f"/content/drive/MyDrive/wazuh-analyst-v2{RUN}"
EXPORT = "/content/export"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%pip -q install aligntune auditkit
from huggingface_hub import login; login()
!git clone $GITHUB_REPO repo 2>/dev/null || (cd repo && git pull)
!ls repo

In [ ]:
# 1 epoch. batch 2 x grad-accum 8 (batch 4 runs out of memory on a 24 GB card: the 262k-vocab logits; same effective batch of 16 as v1's 1 x 16 on the T4); bf16 is picked automatically.
# For a quick check set MAX_STEPS = 3 first, then -1 for the full run.
MAX_STEPS = -1
!cd repo && python train/train_sft.py --dataset $DATASET --subset $SUBSET --out $OUT --epochs 1 --max-steps $MAX_STEPS --batch 2 --grad-accum 8

In [ ]:
# 1) save the adapter first (the runtime can die during export)
from huggingface_hub import HfApi; api = HfApi()
api.create_repo(ADAPTER_REPO, exist_ok=True)
api.upload_folder(folder_path=OUT, repo_id=ADAPTER_REPO,
                  allow_patterns=['adapter_*', '*.json', 'tokenizer*', '*.jinja', 'README.md'], ignore_patterns=['checkpoint-*/**', 'merged/**'])
# 2) merge with AlignTune (it writes to <output>/merged)
!aligntune export merge-adapter $OUT --output $EXPORT/merged
# 3) GGUF with llama.cpp directly (AlignTune's gguf exporter 404s on a renamed converter script)
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install /content/llama.cpp/gguf-py
!mkdir -p $EXPORT/gguf
!python /content/llama.cpp/convert_hf_to_gguf.py $EXPORT/merged/merged --outtype f16 --outfile $EXPORT/gguf/tiny-aya-wazuh-f16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /dev/null
!/content/llama.cpp/build/bin/llama-quantize $EXPORT/gguf/tiny-aya-wazuh-f16.gguf $EXPORT/gguf/tiny-aya-wazuh-analyst-Q4_K_M.gguf Q4_K_M
!rm $EXPORT/gguf/tiny-aya-wazuh-f16.gguf
# 4) upload the GGUF
api.create_repo(GGUF_REPO, exist_ok=True)
api.upload_file(path_or_fileobj=f'{EXPORT}/gguf/tiny-aya-wazuh-analyst-Q4_K_M.gguf', path_in_repo='tiny-aya-wazuh-analyst-Q4_K_M.gguf', repo_id=GGUF_REPO)
print("done:", GGUF_REPO, "- now disconnect the runtime (Runtime > Disconnect and delete runtime)")

Then on the homelab box: `ollama pull hf.co/<HF_USER>/<GGUF_REPO>:Q4_K_M` and run `src/eval.py` with `--model ollama:hf.co/<HF_USER>/<GGUF_REPO>:Q4_K_M` on the same frozen test and attack sets as the baseline.

## Q8_0 export of v1 (TODO #28, no training)
Run cell 2 first (installs, Hugging Face login with a **write** token), then this cell only; skip cells 1, 3 and 4. Any CUDA runtime is fine. It downloads the v1 adapter from Hugging Face, merges it with AlignTune, converts with llama.cpp and quantizes to `Q8_0`, then uploads to a **new** repo. The v1 repos are never written to.

In [ ]:
SRC_ADAPTER = "Yusheng88/tiny-aya-wazuh-analyst-lora"        # v1, read only
Q8_REPO = "Yusheng88/tiny-aya-wazuh-analyst-GGUF-Q8"           # new repo
QUANT = "Q8_0"
ADIR, QX = "/content/adapter-v1", "/content/export-q8"
from huggingface_hub import HfApi, snapshot_download
api = HfApi()
assert Q8_REPO not in ("Yusheng88/tiny-aya-wazuh-analyst-GGUF", SRC_ADAPTER), "never upload into the v1 repos"
snapshot_download(SRC_ADAPTER, local_dir=ADIR, allow_patterns=['adapter_*', '*.json', 'tokenizer*', '*.jinja'])
!ls -la $ADIR
# merge (AlignTune writes to <output>/merged, so the model ends up in $QX/merged/merged)
!aligntune export merge-adapter $ADIR --output $QX/merged
!ls $QX/merged/merged
# llama.cpp convert to f16, then quantize to Q8_0 (same steps as cell 4)
![ -d /content/llama.cpp ] || git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip -q install /content/llama.cpp/gguf-py
!mkdir -p $QX/gguf
!python /content/llama.cpp/convert_hf_to_gguf.py $QX/merged/merged --outtype f16 --outfile $QX/gguf/tiny-aya-wazuh-f16.gguf
![ -x /content/llama.cpp/build/bin/llama-quantize ] || (cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null && cmake --build /content/llama.cpp/build --target llama-quantize -j 8 > /dev/null)
!/content/llama.cpp/build/bin/llama-quantize $QX/gguf/tiny-aya-wazuh-f16.gguf $QX/gguf/tiny-aya-wazuh-analyst-$QUANT.gguf $QUANT
!rm $QX/gguf/tiny-aya-wazuh-f16.gguf; ls -la $QX/gguf
card = f"""---
license: cc-by-nc-4.0
base_model: CohereLabs/tiny-aya-global
tags: [gguf, wazuh, security, lexsi-hackathon]
---
# tiny-aya-wazuh-analyst, {QUANT} GGUF

The same fine-tune as [Yusheng88/tiny-aya-wazuh-analyst-GGUF](https://huggingface.co/Yusheng88/tiny-aya-wazuh-analyst-GGUF) (v1, Q4_K_M),
exported at {QUANT} from the adapter [{SRC_ADAPTER}](https://huggingface.co/{SRC_ADAPTER}) to measure quantization loss.
Not part of the hackathon submission. Trained on synthetic data only. Code: https://github.com/Patupi1/wazuh-alert-analyst

`ollama pull hf.co/{Q8_REPO}:{QUANT}`
"""
open(f"{QX}/gguf/README.md", "w").write(card)
api.create_repo(Q8_REPO, exist_ok=True)
api.upload_file(path_or_fileobj=f"{QX}/gguf/tiny-aya-wazuh-analyst-{QUANT}.gguf", path_in_repo=f"tiny-aya-wazuh-analyst-{QUANT}.gguf", repo_id=Q8_REPO)
api.upload_file(path_or_fileobj=f"{QX}/gguf/README.md", path_in_repo="README.md", repo_id=Q8_REPO)
print("done:", Q8_REPO, "- now disconnect the runtime (Runtime > Disconnect and delete runtime)")